In [ ]:
import os
from google.colab import drive
os.environ["WANDB_DISABLED"] = "true"

In [ ]:
drive.mount("/gdrive", force_remount=True)
main_dir = "/gdrive/MyDrive/ITCS6156_BERT"
result_dir = main_dir + "/Results"
train_path = main_dir + "/Data/train_data.txt"
test_path = main_dir + "/Data/test_data_solution.txt"

!ls {train_path}
!ls {test_path}

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from nltk.tokenize import word_tokenize

#!pip install nltk
import nltk
from nltk.corpus import stopwords
nltk.download('punkt')
nltk.download('stopwords')

import torch
from transformers import AutoTokenizer, DataCollatorWithPadding
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer, EarlyStoppingCallback, AutoConfig
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix

In [ ]:
#pip install transformers==4.45.2 sentence-transformers==3.1.1

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")
# to convert our samples to PyTorch tensors and concatenate them with the correct amount of padding
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [ ]:
def tokenizer_function(df):
    tokenized = tokenizer(df['text'], padding=True, truncation=True)
    tokenized['labels'] = df['labels']

    return tokenized

In [ ]:
def compute_metrics(p):
    predictions, labels = p
    predictions = predictions.argmax(axis=-1)

    accuracy = accuracy_score(labels, predictions)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, predictions, average='weighted', zero_division=1)

    return {
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1
    }

In [ ]:
class CustomTrainer(Trainer):
    def __init__(self, class_weights, *args, **kwargs):
        super().__init__(*args, **kwargs)
        # self.class_weights = class_weights
        self.class_weights = class_weights.to(self.model.device)

    def compute_loss(self, model, inputs, return_outputs=False):
        # labels = inputs.get("labels")
        labels = inputs.get("labels").to(self.model.device)
        # outputs = model(**inputs)
        outputs = model(**{k: v.to(self.model.device) for k, v in inputs.items()})
        logits = outputs.get("logits")

        loss_fct = torch.nn.CrossEntropyLoss(weight=self.class_weights)
        loss = loss_fct(logits, labels)

        return (loss, outputs) if return_outputs else loss

# 1. Load data

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
# Load data
train_path = '/content/drive/MyDrive/ITCS6156_BERT/Data/train_data.txt'

train_data = pd.read_csv(train_path, sep=':::', names=['Title', 'Genre', 'Description'], engine='python')
# test_data = pd.read_csv(test_path, sep=':::', names=['Id', 'Title', 'Genre', 'Description'], engine='python')
# test_data.drop(columns=["Id"], inplace=True)

In [ ]:
train_data

In [ ]:
train_data['Genre'].unique()

In [ ]:
genre_counts = train_data['Genre'].value_counts()
genre_counts

# 2. Data pre-processing

In [ ]:
processed_train_data = train_data.rename(columns={'Description': 'text'})
processed_train_data['text'] = processed_train_data['text'].apply(lambda x: str(x).replace("\xa0", "").lower())
processed_train_data['text'] = processed_train_data['text'].apply(lambda x: x.replace('\n', ' ').replace('\r', ' '))
processed_train_data=processed_train_data.drop_duplicates()

# processed_test_data = test_data.rename(columns={'Description': 'text'})
# processed_test_data['text'] = processed_test_data['text'].apply(lambda x: str(x).replace("\xa0", "").lower())
# processed_test_data['text'] = processed_test_data['text'].apply(lambda x: x.replace('\n', ' ').replace('\r', ' '))
# processed_test_data=processed_test_data.drop_duplicates()

In [ ]:
# Convert text label to numeric label
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()
processed_train_data['labels'] = label_encoder.fit_transform(processed_train_data['Genre'])
# processed_test_data['labels'] = label_encoder.fit_transform(processed_test_data['Genre'])
count_label = len(label_encoder.classes_)

In [ ]:
processed_train_data

In [ ]:
# processed_test_data

In [ ]:
from sklearn.utils.class_weight import compute_class_weight

# For class weight adjustment due to class imbalance problem
labels = processed_train_data['labels'].values
class_weights = compute_class_weight(class_weight='balanced', classes=np.unique(labels), y=labels)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float32).to(device)

In [ ]:
!pip install datasets

In [ ]:
from datasets import Dataset
train_dataset = Dataset.from_pandas(processed_train_data)
# test_dataset = Dataset.from_pandas(processed_test_data)

# 3. Tokenization

In [ ]:
tokenized_train_dataset = train_dataset.map(tokenizer_function, batched=True)
# tokenized_test_dataset = test_dataset.map(tokenizer_function, batched=True)

In [ ]:
df_tk = pd.DataFrame(tokenized_train_dataset)

train_df, test_df = train_test_split(df_tk, test_size=0.2, stratify=df_tk['labels'], random_state=42, shuffle=True)
train_df = train_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)
train = Dataset.from_pandas(train_df)
test = Dataset.from_pandas(test_df)

# 4. Create a BERT model

In [ ]:
config = AutoConfig.from_pretrained("distilbert-base-uncased")
config.hidden_dropout_prob = 0.4
config.num_labels = count_label

model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    config=config
)

# 5. Train the model

In [ ]:
training_args = TrainingArguments(
    output_dir=result_dir,
    learning_rate=7e-5,
    per_device_train_batch_size=20,
    per_device_eval_batch_size=20,
    num_train_epochs=5,
    weight_decay=0.01,
    lr_scheduler_type="cosine",
    save_strategy="epoch",
    eval_strategy="epoch",
    logging_strategy="epoch",
    load_best_model_at_end=True
)

trainer = CustomTrainer(
    model=model,
    args=training_args,
    train_dataset=train,
    eval_dataset=test,
    # train_dataset=tokenized_train_dataset,
    # eval_dataset=tokenized_test_dataset,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    class_weights=class_weights_tensor
)

In [ ]:
trainer.train()

In [ ]:
predictions = trainer.predict(test)
pred_labels = np.argmax(predictions.predictions, axis=-1)

conf_matrix = confusion_matrix(predictions.label_ids, pred_labels)

In [ ]:
plt.figure(figsize=(6, 4))
unique_labels = processed_train_data[['Genre', 'labels']].drop_duplicates()
sorted_labels = unique_labels.sort_values(by='labels')
original_labels = sorted_labels['Genre'].tolist()

sns.heatmap(conf_matrix, annot=True, fmt="d", cmap="Blues", cbar=True,
            xticklabels=original_labels, yticklabels=original_labels)

plt.xticks(fontsize=8, rotation=45, ha='right')
plt.yticks(fontsize=8)
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.title("Confusion Matrix Heatmap")
plt.show()

In [ ]:
trainer.evaluate()